# 01 – Datenladen & Joining

Dieses Notebook lädt alle Rohdatenquellen und führt sie zu einem kombinierten Datensatz zusammen.

**Datenquellen:**
- Smart-Meter-Zeitreihen (`households`)
- Haushalts-Metadaten (`households_info`)
- Wetterdaten (`weather`)
- Vor-Ort-Protokolle (`protocols`)
- Strompreisdaten (`prices`)

In [7]:
%pip install pandera polars


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import sys
import os
import polars as pl
from pathlib import Path

# Pfade setzen
current_dir = Path('..') / '03_src'
sys.path.append(str(current_dir))
root_dir = Path('..')

from config import (
    Schema, fill_false_list, fill_null_list,
    fill_median_list, fill_inter_list,
    fill_string_float, fill_string_boolean,
    fill_string_integer, fill_unbekannt_list,
    target_col, feature_cols, feature_cols_cleaned
)
from data_loader import smartmeter_load, household_load, house_info_load, weather_load, load_price_data
from data_combine import join_data

## 1.1 Pfade definieren

In [9]:
# TODO: Pfade ggf. anpassen
paths = {
    'households': str(root_dir / '02_data' / 'raw' / 'households'),
    'info':       str(root_dir / '02_data' / 'raw' / 'households_info'),
    'weather':    str(root_dir / '02_data' / 'raw' / 'weather'),
    'protocols':  str(root_dir / '02_data' / 'raw' / 'protocols'),
    'prices':     str(root_dir / '02_data' / 'raw' / 'prices')
}

## 1.2 Daten laden

In [10]:
df_smartmeter     = smartmeter_load(data_path=paths['households'])
df_protocols      = house_info_load(data_path=paths['protocols'])
df_household_info = household_load(data_path=paths['info'])
df_prices         = load_price_data(data_path=paths['prices'])
df_weather        = weather_load(data_path=paths['weather'])

print(f"Unique Haushalte Smart Meter:  {df_smartmeter.select(pl.col('household_id').n_unique()).item()}")
print(f"Unique Haushalte Protokolle:   {df_protocols.select(pl.col('household_id').n_unique()).item()}")
print(f"Unique Haushalte Metadaten:    {df_household_info.select(pl.col('household_id').n_unique()).item()}")

✅ Price Data geladen: 2272 Zeilen, 2 Spalten.
Unique Haushalte Smart Meter:  1298
Unique Haushalte Protokolle:   215
Unique Haushalte Metadaten:    1408


## 1.3 Daten joinen

Alle Quellen werden über `household_id` und `date` als Left-Join zusammengeführt.

In [11]:
data_combined = join_data(
    df_smartmeter,
    joins=[
        {'df': df_household_info, 'on': ['household_id'], 'how': 'left'},
        {'df': df_weather, 'on': ['weather_id', 'date'], 'how': 'left'},
        {'df': df_protocols, 'left_on': ['household_id'], 'right_on': ['household_id_info'], 'how': 'left'},
        {'df': df_prices, 'on': ['date'], 'how': 'left'},
    ]
)

print(f"Unique Haushalte Combined: {data_combined.select(pl.col('household_id').n_unique()).item()}")
print(f"Beobachtungen Combined:    {data_combined.select(pl.col('household_id').count()).item()}")
data_combined.head()

Unique Haushalte Combined: 1298
Beobachtungen Combined:    937456


timestamp,timestamp_local,date,year_str,household_id,group_assignment,affects_timepoint,kwh_received_total,kwh_received_heatpump,kwh_received_other,kwh_returned_total,group,weather_id,installation_haspvsystem,protocols_available,protocols_hasmultiplevisits,protocols_reportids,metadata_available,smartmeterdata_available_15min,smartmeterdata_available_daily,smartmeterdata_available_monthly,temperature_avg_daily,temperature_max_daily,temperature_min_daily,heatingdegree_sia_daily,heatingdegree_us_daily,coolingdegree_us_daily,humidity_avg_daily,precipitation_total_daily,sunshine_duration_daily,timestamp_local_right,report_id,household_id_right,visit_year,visit_date,building_type,building_housingunits,…,heatpump_groundsource_currentpressure,heatpump_groundsource_currentpressure_okay,heatpump_groundsource_currenttemperature,heatpump_groundsource_currenttemperature_okay,heatpump_heatingcurvesetting_toohigh_beforevisit,heatpump_heatingcurvesetting_changed,heatpump_heatingcurvesetting_outside20_beforevisit,heatpump_heatingcurvesetting_outside0_beforevisit,heatpump_heatingcurvesetting_outsideminus8_beforevisit,heatpump_heatingcurvesetting_outside20_aftervisit,heatpump_heatingcurvesetting_outside0_aftervisit,heatpump_heatingcurvesetting_outsideminus8_aftervisit,heatpump_heatinglimitsetting_toohigh_beforevisit,heatpump_heatinglimitsetting_changed,heatpump_heatinglimitsetting_beforevisit,heatpump_heatinglimitsetting_aftervisit,heatpump_nightsetbacksetting_activated_beforevisit,heatpump_nightsetbacksetting_activated_aftervisit,dhw_temperaturesetting_categorization,dhw_temperaturesetting_changed,dhw_temperaturesetting_beforevisit,dhw_temperaturesetting_aftervisit,dhw_storage_lastdescaling_toolongago,dhw_storage_lastdescaling_year,heatdistribution_expansiontank_pressure_categorization,heatdistribution_expansiontank_pressure_actual,heatdistribution_expansiontank_pressure_target,heatdistribution_expansiontank_systemheight,heatdistribution_circulation_pumpstageposition_changed,heatdistribution_circulation_pumpstageposition_beforevisit,heatdistribution_circulation_pumpstageposition_aftervisit,heatdistribution_recommendation_insulatepipes,heatdistribution_recommendation_installthermostaticvalve,heatdistribution_recommendation_installrpmvalve,visit_date_date,year_str_right,swissix_base
"datetime[μs, UTC]","datetime[μs, Europe/Zurich]",date,str,str,str,str,f64,f64,f64,f64,str,str,bool,bool,bool,str,bool,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,"datetime[μs, Europe/Zurich]",i64,i64,i64,str,str,f64,…,f64,bool,f64,bool,bool,bool,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,bool,bool,str,bool,f64,f64,bool,f64,str,f64,f64,f64,bool,f64,f64,bool,bool,bool,date,str,f64
2019-03-02 23:59:59 UTC,2019-03-03 00:59:59 CET,2019-03-03,"""2019""","""100101""","""control""","""unknown""",18.33,null,null,8.64,"""control""","""z6I""",true,false,false,null,true,true,true,true,8.4,13.8,2.9,11.6,9.9,0.0,65.9,0.0,6.3,2019-03-03 01:00:00 CET,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,22.79
2019-03-03 23:59:59 UTC,2019-03-04 00:59:59 CET,2019-03-04,"""2019""","""100101""","""control""","""unknown""",15.03,null,null,9.04,"""control""","""z6I""",true,false,false,null,true,true,true,true,7.8,13.8,5.0,12.2,10.5,0.0,54.2,2.6,1.3,2019-03-04 01:00:00 CET,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,37.938333
2019-03-04 23:59:59 UTC,2019-03-05 00:59:59 CET,2019-03-05,"""2019""","""100101""","""control""","""unknown""",16.69,null,null,4.57,"""control""","""z6I""",true,false,false,null,true,true,true,true,7.4,11.9,2.7,12.6,10.9,0.0,55.5,0.2,7.7,2019-03-05 01:00:00 CET,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,

In [15]:
def compare_households(source_df, combined_df, source_name, id_col_source, id_col_combined="household_id"):
    
    source_ids = set(source_df.select(pl.col(id_col_source).unique()).to_series().to_list())
    combined_ids = set(combined_df.select(pl.col(id_col_combined).unique()).to_series().to_list())
    
    matched = source_ids.intersection(combined_ids)
    lost = source_ids.difference(combined_ids)
    
    print(f"{source_name}")
    print(f"  Haushalte im Ausgangsdatensatz: {len(source_ids)}")
    print(f"  Haushalte im Combined:          {len(matched)}")
    print(f"  Nicht gematcht / verloren:      {len(lost)}")
    print(f"  Match-Rate:                     {len(matched)/len(source_ids):.2%}")
    print("-" * 50)


compare_households(df_smartmeter, data_combined, "Smartmeter", "household_id")
compare_households(df_household_info, data_combined, "Metadaten", "household_id")
compare_households(df_protocols, data_combined, "Protokolle", "household_id_info")

Smartmeter
  Haushalte im Ausgangsdatensatz: 1298
  Haushalte im Combined:          1298
  Nicht gematcht / verloren:      0
  Match-Rate:                     100.00%
--------------------------------------------------
Metadaten
  Haushalte im Ausgangsdatensatz: 1408
  Haushalte im Combined:          1298
  Nicht gematcht / verloren:      110
  Match-Rate:                     92.19%
--------------------------------------------------
Protokolle
  Haushalte im Ausgangsdatensatz: 215
  Haushalte im Combined:          156
  Nicht gematcht / verloren:      59
  Match-Rate:                     72.56%
--------------------------------------------------


## 1.4 Zwischenspeichern

In [13]:
output_path_temp = root_dir / "02_data" / "temp" / "combined_data.parquet"

output_path_temp.parent.mkdir(parents=True, exist_ok=True)
data_combined.write_parquet(output_path_temp)
print(f'✅ Gespeichert: {output_path}')

✅ Gespeichert: ..\02_data\temp\combined_raw.csv
